# Structured Output Extraction – JSON Mode with Groq

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
This notebook uses Groq’s Llama 3.3 70B to extract structured data (JSON) from unstructured text. You provide a JSON schema describing the fields you want, and the model returns a JSON object with the extracted values. This is a production‑grade technique for turning free‑form text into database‑ready records.

## What You Will Build
- A function that sends a prompt with a JSON schema.
- Automatic parsing and validation of JSON responses.
- Example schemas: invoice extraction, resume data, customer feedback.
- Interactive mode – define any schema on the fly.

## Why This Matters
Most real‑world data is unstructured. JSON extraction allows LLMs to feed structured information into dashboards, spreadsheets, or databases. This skill is highly valued in data engineering and automation.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Structuring unstructured text with LLMs.**

### Install & Imports

In [1]:
!pip install -q groq

import json
from getpass import getpass
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.4 MB/s eta 0:00:00


### API Key & Groq Client

In [2]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### JSON Extraction Function

In [3]:
def extract_json(text, schema_description, field_definitions):
    """
    Extract structured data from text using a JSON schema.
    schema_description: plain English description of what to extract.
    field_definitions: dict of field names and descriptions.
    """
    prompt = f"""Extract the requested fields from the text below and output a valid JSON object.
Do not include any explanations or extra text – only output JSON.

Schema description: {schema_description}
Fields:
{json.dumps(field_definitions, indent=2)}

Example output format:
{{
    "field1": "value1",
    "field2": "value2"
}}

Text:
{text}

JSON output:
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1024,
        response_format={"type": "json_object"}  # forces JSON output
    )
    content = response.choices[0].message.content
    try:
        data = json.loads(content)
        return True, data
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}\nRaw output: {content}"

### Example 1: Invoice Extraction

In [4]:
invoice_text = """
INVOICE #INV-2026-001
Date: 2026-05-11
Customer: Acme Corp
Items:
- 10 x Widgets @ $5.00 each
- 2 x Gadgets @ $25.00 each
Total: $100.00
"""

schema_desc = "Extract invoice details including number, date, customer, line items, and total."
fields = {
    "invoice_number": "The invoice identifier",
    "date": "Issue date in YYYY-MM-DD format",
    "customer": "Customer name",
    "line_items": "List of items with quantity, description, unit price",
    "total": "Total amount as a number"
}

success, result = extract_json(invoice_text, schema_desc, fields)
if success:
    print(" Extracted Data:")
    print(json.dumps(result, indent=2))
else:
    print(result)

 Extracted Data:
{
  "invoice_number": "INV-2026-001",
  "date": "2026-05-11",
  "customer": "Acme Corp",
  "line_items": [
    {
      "quantity": 10,
      "description": "Widgets",
      "unit_price": 5.0
    },
    {
      "quantity": 2,
      "description": "Gadgets",
      "unit_price": 25.0
    }
  ],
  "total": 100.0
}


### Example 2: Resume/Candidate Extraction

In [5]:
resume_text = """
John Doe
Email: john.doe@example.com
Phone: +1 (555) 123-4567
Skills: Python, SQL, Machine Learning
Experience:
- Data Scientist at ABC Corp (2020-2024): built recommendation models.
- Data Analyst at XYZ Ltd (2018-2020): created dashboards.
"""

schema_desc = "Extract candidate information from a resume."
fields = {
    "full_name": "Candidate's full name",
    "email": "Email address",
    "phone": "Phone number",
    "skills": "List of technical skills",
    "experience": "List of job titles and companies"
}

success, result = extract_json(resume_text, schema_desc, fields)
if success:
    print(" Extracted Data:")
    print(json.dumps(result, indent=2))
else:
    print(result)

 Extracted Data:
{
  "full_name": "John Doe",
  "email": "john.doe@example.com",
  "phone": "+1 (555) 123-4567",
  "skills": [
    "Python",
    "SQL",
    "Machine Learning"
  ],
  "experience": [
    {
      "job_title": "Data Scientist",
      "company": "ABC Corp",
      "years": "2020-2024"
    },
    {
      "job_title": "Data Analyst",
      "company": "XYZ Ltd",
      "years": "2018-2020"
    }
  ]
}


### Example 3: Customer Feedback Sentiment

In [6]:
feedback = """
I bought the wireless mouse and it stopped working after a week. Very disappointed.
The customer service was slow to respond. Would not recommend.
"""

schema_desc = "Extract sentiment and key complaints from customer feedback."
fields = {
    "sentiment": "positive, neutral, or negative",
    "product_mentioned": "Name of the product",
    "complaint": "Main issue described",
    "recommendation": "Would they recommend? (yes/no/unknown)"
}

success, result = extract_json(feedback, schema_desc, fields)
if success:
    print(" Extracted Data:")
    print(json.dumps(result, indent=2))
else:
    print(result)

 Extracted Data:
{
  "sentiment": "negative",
  "product_mentioned": "wireless mouse",
  "complaint": "stopped working after a week, slow customer service response",
  "recommendation": "no"
}


### Interactive Mode: Define Your Own Schema

In [7]:
print("Interactive JSON Extraction")
print("Provide your own text and schema.")
print("Type 'exit' to quit the interactive loop.\n")

while True:
    print("\n--- New Extraction ---")
    text = input(" Text (or 'exit'): ").strip()
    if text.lower() == "exit":
        break
    if not text:
        continue
    schema_desc = input(" Schema description: ").strip()
    print(" Define fields (field name: description). Type 'done' when finished.")
    fields = {}
    while True:
        line = input("  field name and description: ").strip()
        if line.lower() == "done":
            break
        if ":" in line:
            name, desc = line.split(":", 1)
            fields[name.strip()] = desc.strip()
        else:
            print("  Format: field_name: description")
    if not fields:
        print("No fields defined. Skipping.")
        continue
    success, result = extract_json(text, schema_desc, fields)
    if success:
        print("\n Extracted JSON:")
        print(json.dumps(result, indent=2))
    else:
        print(f"\n {result}")

Interactive JSON Extraction
Provide your own text and schema.
Type 'exit' to quit the interactive loop.


--- New Extraction ---
 Text (or 'exit'): Dear Sir/Madam,  My name is Alice Johnson. My order #ORD-98765 was placed on 2026-05-10 for a "Wireless Headphones" costing $79.99. Unfortunately the product arrived damaged. I would like a full refund.  Sincerely, Alice
 Schema description: Extract customer details from a refund request email.
 Define fields (field name: description). Type 'done' when finished.
  field name and description: customer_name: Name of the customer order_number: Order identifier date: Date the order was placed product: Name of the product amount: Price as a number issue: Main problem described requested_action: What the customer wants
  field name and description: done

 Extracted JSON:
{
  "customer_name": "Alice Johnson",
  "order_number": "ORD-98765",
  "date": "2026-05-10",
  "product": "Wireless Headphones",
  "amount": 79.99,
  "issue": "damaged product",


###  Final Summary

In [8]:
print("Structured Output Extraction - COMPLETED")
print("Author: Ibrahim")
print(" Extract JSON from unstructured text using Groq's JSON mode.")
print(" Support for custom schemas.")
print(" Examples: invoice, resume, feedback.")
print(" Ready for data pipelines and automation.")

Structured Output Extraction - COMPLETED
Author: Ibrahim
 Extract JSON from unstructured text using Groq's JSON mode.
 Support for custom schemas.
 Examples: invoice, resume, feedback.
 Ready for data pipelines and automation.
